# Study of the 50 DOF / 21 vmode LUT (BLOCK-T710) (v1)

**Author:** Aaron Roodman  
**Date Created:** 2026-05-05  
**Last Modified:** 2026-05-05  
**Status:** Draft  
**Keywords:** AOS, FAM, LUT, 50 DOF, 21 vmode, BLOCK-T710  

## Description

Analyse BLOCK-T710 — three FAM blocks of 9 triplets each in which the
rotator stepped from 0 to 15 deg per block.  The three blocks are:

1. **`dz_nolut`**       — FAM with neither the LUT nor an alignment update.
2. **`dz_lut`**         — FAM with the 50 DOF / 21 vmode LUT applied.
3. **`dz_align_lut`**   — FAM after a normal in-focus closed-loop alignment
                            *and* the LUT applied.

Compare the per-(k, j) DZ coefficient values of the three blocks against
Theo's expectation:

* the median DZ from the rot=0 / alt=70 deg subset of the long
  Mar 15 - Apr 9 measurement campaign — `dzm_nolut`;
* the residual DZ predicted after the 50 DOF / 21 vmode LUT —
  `dzm_lut`;
* their difference — `dzm_delta = dzm_nolut - dzm_lut`.

**Output:** PDF of per-pupil-j pages and a parquet table summarising the
median DZ values per block, in `output/study_50dofLUT/`.  
**Based on:** the standard `fit_params_resid_z1toz6_all.pdf` style; Theo's
`dz_coefficients_*` and `dz_residuals_*` `.npy` files.

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-05-05 | Aaron Roodman | Initial version |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access — BLOCK-T710 fit subset](#data)
5. [Theo's expectation — median + LUT residual](#theo)
6. [Per-pupil-j comparison plots](#plots)
7. [Output Tables](#output)

<a id='params'></a>
## 1. Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# ----- Input fit parquet -----
fits_parquet_path = (
    'output/aos_fam_danish_v1_triplets_20260418_20260423_fits.parquet')

# Fit prefix (z1toz3 or z1toz6).
fit_prefix = 'z1toz6'

# ----- BLOCK-T710 selection -----
# NOTE: the upstream pipeline (intrinsics_lib.py) does not yet copy
# `science_program` or `reason` from ConsDB into the fits parquet, so
# we identify the BLOCK-T710 visits by day_obs + chronological order.
# All 27 visits land on a single day; the 3 sub-blocks of 9 are
# contiguous in seq_num.
block_day_obs        = 20260423
block_seq_num_range  = None     # e.g. (21, 115); None = use whole day

# 9 triplets per sub-block, 3 sub-blocks total -> 27 visits expected.
n_per_subblock = 9

# Sub-block labels and styling.
subblock_labels  = ('no LUT', 'with LUT', 'aligned + LUT')
subblock_keys    = ('dz_nolut', 'dz_lut', 'dz_align_lut')
subblock_markers = ('o', 's', '^')
subblock_colors  = ('tab:blue', 'tab:orange', 'tab:green')

# ----- Theo's expected values -----
# Both files contain shape [1, 6, n_j] arrays.  The j axis order
# matches `nollIndices` from the fits parquet.  Defaults to
# auto-derive from that column.
theo_median_npy   = (
    'output/50DOF21vmode_LUT/'
    'dz_coefficients_22dof_geom_gq_rank10_danish1.0_0315-0409_rot0_alt70_v1.npy')
theo_residual_npy = (
    'output/50DOF21vmode_LUT/'
    'dz_residuals_50dof_geom_gq_rank21_danish1.0_0315-0409_rot0_alt70_v1.npy')
theo_j_indices    = None        # None -> use nollIndices from the fits parquet
theo_k_range      = range(1, 7)

# ----- Plot range and styling -----
# Pupil Zernike pages: None -> use nollIndices from the fits parquet
# (Z4..Z19 + Z22..Z26 = 21 entries; Z20 / Z21 are absent upstream).
pupil_j_range = None
# Focal-plane Zernike panels per page (k = 1..6).
focal_k_range = range(1, 7)

# ----- Output -----
output_dir         = 'output/study_50dofLUT'
output_pdf         = f'{output_dir}/study_50dofLUT_perj.pdf'
output_table_path  = f'{output_dir}/study_50dofLUT_dz_table.parquet'

<a id='setup'></a>
## 2. Setup & Imports

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from astropy.table import QTable

sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting

setup_plotting()

<a id='functions'></a>
## 3. Helper Functions

In [ ]:
FOCAL_NAMES = {
    1: 'Piston', 2: 'Tilt', 3: 'Tip',
    4: 'Focus', 5: 'Astig45', 6: 'Astig0',
}
PUPIL_NAMES = {
    4: 'Defocus', 5: 'Astig45', 6: 'Astig0',
    7: 'Coma_y', 8: 'Coma_x', 9: 'Trefoil_y', 10: 'Trefoil_x',
    11: 'Spherical', 12: '2ndAstig0', 13: '2ndAstig45',
    14: 'Tetrafoil_x', 15: 'Tetrafoil_y',
    16: '2ndComa_x', 17: '2ndComa_y',
    18: '2ndTrefoil_x', 19: '2ndTrefoil_y',
    20: 'Pentafoil_x', 21: 'Pentafoil_y',
    22: '2ndSpherical', 23: '3rdAstig45', 24: '3rdAstig0',
    25: '4thTrefoil_y', 26: '4thTrefoil_x',
}


def select_block_visits(fit_table, day_obs, seq_num_range=None):
    """Boolean mask for visits on a given day, optionally restricted to
    a seq_num range (inclusive)."""
    dobs = np.asarray(fit_table['day_obs']).astype(int)
    snum = np.asarray(fit_table['seq_num']).astype(int)
    mask = (dobs == int(day_obs))
    if seq_num_range is not None:
        lo, hi = int(seq_num_range[0]), int(seq_num_range[1])
        mask &= (snum >= lo) & (snum <= hi)
    return mask


def split_into_subblocks(fit_block, n_per_subblock):
    """Sort by seq_num and split into 3 sub-blocks of n_per_subblock each.

    Returns three index arrays (positions inside fit_block).
    """
    snum = np.asarray(fit_block['seq_num']).astype(int)
    order = np.argsort(snum)
    n_total = len(order)
    expected = 3 * n_per_subblock
    if n_total != expected:
        print(f'  WARNING: BLOCK-T710 has {n_total} visits but expected '
              f'{expected} (3 sub-blocks of {n_per_subblock}).')
    sub = []
    for i in range(3):
        start = i * n_per_subblock
        end   = (i + 1) * n_per_subblock
        sub.append(order[start:end])
    return sub


def safe_load_npy(path):
    p = Path(path)
    if not p.exists():
        print(f'  WARNING: missing {p}; LUT lines/arrows will be omitted.')
        return None
    arr = np.load(str(p))
    print(f'  Loaded {p.name}  shape={arr.shape}')
    return arr


def add_lut_overlay(ax, x_lo, x_hi, dzm_med, dzm_res):
    """Two horizontal lines + an arrow from `dzm_med` to `dzm_res`.

    Pads the y-limits if needed so both lines are inside the axes.
    """
    ax.hlines(dzm_med, x_lo, x_hi, color='black',
              lw=1.5, alpha=0.85, label=f'Theo med = {dzm_med:+.3f}')
    ax.hlines(dzm_res, x_lo, x_hi, color='dimgray',
              lw=1.5, ls='--', alpha=0.85,
              label=f'Theo lut residual = {dzm_res:+.3f}')
    x_arrow = 0.5 * (x_lo + x_hi)
    ax.annotate('', xy=(x_arrow, dzm_res), xytext=(x_arrow, dzm_med),
                arrowprops=dict(arrowstyle='->', color='black',
                                lw=1.4, alpha=0.85))
    y_lo, y_hi = ax.get_ylim()
    span = max(abs(dzm_med), abs(dzm_res), abs(dzm_med - dzm_res), 1e-3)
    pad = 0.10 * span
    new_lo = min(y_lo, dzm_med, dzm_res) - pad
    new_hi = max(y_hi, dzm_med, dzm_res) + pad
    ax.set_ylim(new_lo, new_hi)


def plot_pupil_j_page(j, fit_table_block, ordinal, sub_indices,
                      sub_labels, sub_markers, sub_colors,
                      focal_k_range, fit_prefix,
                      theo_med, theo_res, theo_j_to_idx):
    """One page (2x3 grid) for pupil j, x-axis = visit ordinal index."""
    fig, axes = plt.subplots(2, 3, figsize=(15, 9), layout='constrained')
    axes = axes.ravel()
    pname = PUPIL_NAMES.get(j, f'Z{j}')

    for k_pos, k in enumerate(focal_k_range):
        ax = axes[k_pos]
        col     = f'{fit_prefix}_z{j}_c{k}'
        err_col = f'{fit_prefix}_z{j}_c{k}_err'
        if col not in fit_table_block.colnames:
            ax.set_visible(False)
            continue
        y_full   = np.asarray(fit_table_block[col],   dtype=float)
        err_full = (np.asarray(fit_table_block[err_col], dtype=float)
                    if err_col in fit_table_block.colnames else None)

        for sub_idx, label, marker, color in zip(
                sub_indices, sub_labels, sub_markers, sub_colors):
            if len(sub_idx) == 0:
                continue
            xs = ordinal[sub_idx]
            ys = y_full[sub_idx]
            es = err_full[sub_idx] if err_full is not None else None
            med_y = float(np.nanmedian(ys))
            ax.errorbar(xs, ys, yerr=es,
                        fmt=marker, color=color, markersize=6,
                        linestyle='none', elinewidth=0.6, capsize=0,
                        alpha=0.85,
                        label=f'{label}  med={med_y:+.3f}')
            ax.axhline(med_y, color=color, ls=':', lw=0.8, alpha=0.45)

        x_lo = float(np.nanmin(ordinal))
        x_hi = float(np.nanmax(ordinal))

        if (theo_med is not None and theo_res is not None
                and j in theo_j_to_idx):
            j_idx = theo_j_to_idx[j]
            dzm_med = float(theo_med[0, k - 1, j_idx])
            dzm_res = float(theo_res[0, k - 1, j_idx])
            add_lut_overlay(ax, x_lo, x_hi, dzm_med, dzm_res)

        ax.axhline(0, color='gray', lw=0.5, alpha=0.5, zorder=0)
        ax.set_xlabel('image ordinal (chronological)', fontsize=9)
        ax.set_ylabel('coeff [μm]', fontsize=9)
        ax.set_title(f'k={k} ({FOCAL_NAMES.get(k, "?")})', fontsize=10)
        ax.tick_params(labelsize=8)
        if k_pos == 0:
            ax.legend(fontsize=7, loc='best')

    fig.suptitle(
        f'Pupil Z{j} ({pname})  —  {fit_prefix}, BLOCK-T710  '
        f'(black = Theo median; dashed gray = Theo LUT residual)',
        fontsize=12)
    return fig


def build_dz_summary_table(fit_table_block, sub_indices,
                            pupil_j_range, focal_k_range, fit_prefix,
                            theo_med, theo_res, theo_j_to_idx,
                            subblock_keys):
    """Long-format summary: one row per (k, j)."""
    rows = []
    for j in pupil_j_range:
        for k in focal_k_range:
            col = f'{fit_prefix}_z{j}_c{k}'
            if col not in fit_table_block.colnames:
                continue
            row = {'k': int(k), 'j': int(j)}
            for key, sub_idx in zip(subblock_keys, sub_indices):
                vals = np.asarray(fit_table_block[col],
                                  dtype=float)[sub_idx]
                row[key] = (float(np.nanmedian(vals)) if len(vals) > 0
                            else np.nan)

            if (theo_med is not None and theo_res is not None
                    and j in theo_j_to_idx):
                j_idx = theo_j_to_idx[j]
                row['dzm_nolut'] = float(theo_med[0, k - 1, j_idx])
                row['dzm_lut']   = float(theo_res[0, k - 1, j_idx])
                row['dzm_delta'] = row['dzm_nolut'] - row['dzm_lut']
            else:
                row['dzm_nolut'] = np.nan
                row['dzm_lut']   = np.nan
                row['dzm_delta'] = np.nan
            rows.append(row)
    return pd.DataFrame(rows)

<a id='data'></a>
## 4. Data Access — BLOCK-T710 fit subset

In [ ]:
fits_path = Path(fits_parquet_path)
if not fits_path.exists():
    raise FileNotFoundError(fits_path)

fit_table = QTable.read(str(fits_path))
print(f'Loaded {fits_path.name}: {len(fit_table)} rows, '
      f'{len(fit_table.colnames)} columns')

# Restrict to the BLOCK-T710 day_obs (and optional seq_num range).
block_mask = select_block_visits(fit_table, block_day_obs,
                                  block_seq_num_range)
fit_block = fit_table[block_mask]
print(f'BLOCK-T710 visits on day_obs={block_day_obs}: {len(fit_block)}')
if 'rotator_angle' in fit_block.colnames:
    rot = np.asarray(fit_block['rotator_angle'], dtype=float)
    print(f'  rotator_angle: {np.nanmin(rot):+.2f} .. {np.nanmax(rot):+.2f} deg')
if 'alt' in fit_block.colnames:
    alt = np.asarray(fit_block['alt'], dtype=float)
    if np.nanmax(np.abs(alt)) < 2 * np.pi + 1e-3:
        alt = np.rad2deg(alt)
    print(f'  elevation:     {np.nanmin(alt):+.2f} .. {np.nanmax(alt):+.2f} deg')

# Drop bad-fit visits if the column is present (matches run_pipeline cuts).
for bf in (f'{fit_prefix}_bad_fit', 'bad_fit'):
    if bf in fit_block.colnames:
        bad = np.asarray(fit_block[bf]).astype(bool)
        n_bad = int(bad.sum())
        if n_bad > 0:
            print(f'  Dropping {n_bad} bad-flagged visits ({bf})')
        fit_block = fit_block[~bad]
        break

# Sort fit_block chronologically (by day_obs, seq_num) and assign ordinals.
fb_dobs = np.asarray(fit_block['day_obs']).astype(int)
fb_snum = np.asarray(fit_block['seq_num']).astype(int)
order = np.lexsort((fb_snum, fb_dobs))
fit_block = fit_block[order]
fb_dobs = fb_dobs[order]
fb_snum = fb_snum[order]
ordinal = np.arange(len(fit_block))

# Split chronologically into 3 sub-blocks of n_per_subblock visits each.
sub_indices = split_into_subblocks(fit_block, n_per_subblock)

# ----- Derive the pupil-Noll j list from the fits parquet -----
if pupil_j_range is None:
    if 'nollIndices' not in fit_block.colnames:
        raise ValueError(
            "fits parquet has no 'nollIndices' column; set pupil_j_range "
            "explicitly in the parameters cell.")
    iZs = [int(j) for j in np.asarray(fit_block['nollIndices'][0]).tolist()]
else:
    iZs = list(pupil_j_range)

if theo_j_indices is None:
    theo_j_list = list(iZs)
else:
    theo_j_list = list(theo_j_indices)

print(f'\nPupil Noll j (from nollIndices): {iZs}  (n={len(iZs)})')
if theo_j_list != iZs:
    print(f"  NOTE: theo_j_indices was set explicitly to {theo_j_list}; "
          f"will be used for the LUT overlay only.")

print('\nSub-block contents:')
for label, idx in zip(subblock_labels, sub_indices):
    if len(idx) == 0:
        print(f'  {label:<18s}  EMPTY')
        continue
    print(f'  {label:<18s}  n={len(idx)}  '
          f'ordinal=[{int(ordinal[idx].min())}..{int(ordinal[idx].max())}]  '
          f'seq={list(fb_snum[idx])}')

<a id='theo'></a>
## 5. Theo's expectation — median + LUT residual

Two `.npy` files of shape `[1, 6, 21]`.  The j axis is
`[4..19, 22..26]` (i.e. Z4..Z26 minus Z20, Z21).

In [ ]:
theo_med = safe_load_npy(theo_median_npy)
theo_res = safe_load_npy(theo_residual_npy)
theo_j_to_idx = {j: i for i, j in enumerate(theo_j_list)}
print(f'Theo j_indices ({len(theo_j_list)}): {theo_j_list}')
if theo_med is not None and theo_res is not None:
    n_k_expect = len(list(theo_k_range))
    n_j_expect = len(theo_j_list)
    if theo_med.shape[-2:] != (n_k_expect, n_j_expect):
        print(f'  WARNING: theo_med shape {theo_med.shape} '
              f'!= (?, {n_k_expect}, {n_j_expect})')
    if theo_res.shape != theo_med.shape:
        print(f'  WARNING: theo_res shape {theo_res.shape} '
              f'!= theo_med shape {theo_med.shape}')

<a id='plots'></a>
## 6. Per-pupil-j comparison plots

One page per pupil j (Z4..Z26).  Each page has a 2x3 grid for focal
k = 1..6.  Three filled markers identify the three sub-blocks; a thin
dotted horizontal line marks the median of each sub-block.  When Theo
has a value at this j: a solid black line is the **median DZ** prior to
any LUT, a dashed gray line is the **predicted LUT residual**, and the
arrow shows the LUT delta (`dzm_delta = dzm_nolut - dzm_lut`).

In [ ]:
Path(output_dir).mkdir(parents=True, exist_ok=True)

n_pages = 0
with PdfPages(output_pdf) as pdf:
    for j in iZs:
        if not any(f'{fit_prefix}_z{j}_c{k}' in fit_block.colnames
                   for k in focal_k_range):
            continue
        fig = plot_pupil_j_page(
            j, fit_block, ordinal, sub_indices,
            subblock_labels, subblock_markers, subblock_colors,
            focal_k_range, fit_prefix,
            theo_med, theo_res, theo_j_to_idx)
        pdf.savefig(fig, bbox_inches='tight')
        if n_pages == 0:
            plt.show()
        plt.close(fig)
        n_pages += 1
print(f'Wrote {n_pages} per-pupil-j pages to {output_pdf}')

<a id='output'></a>
## 7. Output Tables

Long-format DZ summary: one row per `(k, j)`.  Columns:

| col | meaning |
|---|---|
| `k`            | focal Noll index 1..6 |
| `j`            | pupil Noll index 4..26 |
| `dz_nolut`     | median across the no-LUT sub-block (this run) |
| `dz_lut`       | median across the with-LUT sub-block (this run) |
| `dz_align_lut` | median across the aligned + LUT sub-block (this run) |
| `dzm_nolut`    | Theo's prior median (Mar 15 - Apr 9, rot=0/alt=70) |
| `dzm_lut`      | Theo's predicted LUT residual |
| `dzm_delta`    | `dzm_nolut - dzm_lut` (the predicted LUT change) |

In [ ]:
df_dz = build_dz_summary_table(
    fit_block, sub_indices,
    iZs, focal_k_range, fit_prefix,
    theo_med, theo_res, theo_j_to_idx,
    subblock_keys)

df_dz.to_parquet(output_table_path)
print(f'Saved DZ summary parquet: {output_table_path}\n'
      f'  {len(df_dz)} rows x {len(df_dz.columns)} columns')

# Display the entire long-format table sorted by (j, k).
df_dz_sorted = df_dz.sort_values(['j', 'k']).reset_index(drop=True)
with pd.option_context('display.max_rows', None,
                       'display.float_format', '{:+.4f}'.format):
    display(df_dz_sorted)